# SRM Composite Models

This notebook runs the primary SRM Global Linear progression-biomarker model.

**Why this model is used**

| Model | Nature | Input | Training target | Output | Interpretation |
|---|---|---|---|---|---|
| SRM Global Linear | One global linear imaging composite | All imaging features | Maximise training-fold Standardized Response Mean (SRM) | One visit score | A participant's imaging progression is the follow-up score minus baseline score. |

**Validation rule:** split by `subject`, not by `pair_id`, so `V1V2` and `V2V3` intervals from the same participant are never separated across train/test.

**Patient-Adaptive/interaction modelling has been moved to `interaction_term.ipynb`.**


In [6]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time, modelling_pair_count_table
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.clinical_validity import clinical_validity
from src.eval.metrics import (
    bootstrap_ci_d,
    bootstrap_paired_metric,
    clinical_change_effect_sizes,
    compute_longitudinal_deltas,
    paired_cohens_dz,
    probability_positive_change,
    reference_effect_sizes,
)
from src.eval.intervals import adjacent_pair_interval_effect_summary, annual_tuning_diagnostics, interval_effect_summary, pooled_adjacent_pair_effect_summary
from src.eval.single_feature import single_feature_interval_baselines
from src.eval.model_selection import select_hierarchical_candidate
from src.reporting.model_performance import assemble_performance_rows, save_one_performance_csv
from src.reporting.tables import final_model_performance_matrix
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.reporting.tuning_review import tuning_recommendation, tuning_verification_summary
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv, srm_global_nested_loocv, srm_global_repeated_group_cv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")
pairs_df = pd.read_csv(pairs_path)

modelling_counts = modelling_pair_count_table(pairs_df, expected={"N12": 108, "N23": 99, "N13": 90, "N123": 90})
print("Canonical modelling-cohort pair counts")
display(modelling_counts)
long_df = trackfa_pairs_to_long(pairs_df)
subject_long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
subject_long_df = add_visit_time(pd.read_csv(subject_long_path)) if subject_long_path.exists() else pd.DataFrame()
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300
RANDOM_SEED = DEFAULT_CONFIG.random_state
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} pair-interval visit rows, {len(imaging_cols)} imaging features")
if not subject_long_df.empty:
    print(f"Loaded {subject_long_path.name}: {subject_long_df.shape[0]} subject-level visit rows")
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


Canonical modelling-cohort pair counts


,count,interval,n,definition
0,N12,V1->V2,108,subjects with a V1V2 annual pair row
1,N23,V2->V3,99,subjects with a V2V3 annual pair row
2,N13,V1->V3,90,subjects with both V1V2 and V2V3 annual pair rows
3,N123,"V1,V2,V3",90,subjects represented across all three visits v...


Loaded trackfa_pairs_drop3poms.csv: 414 pair-interval visit rows, 146 imaging features
Loaded trackfa_long.csv: 522 subject-level visit rows


## 1. Experiment Settings

This cell defines the shared SRM modelling configuration: all imaging features, grouped cross-validation strategy, random seed, and SRM regularisation settings.

Regularisation is used to reduce overfitting and estimator instability. In this notebook, the key regularisation idea is covariance shrinkage for SRM.


In [7]:
# Experiment settings: primary analyses use the full imaging pool.
selection_method = "none"
selection_k = int(globals().get("selection_k", 8))
# SRM uses grouped CV with participant-level splitting.
CV_N_SPLITS = globals().get("CV_N_SPLITS", DEFAULT_CONFIG.cv_n_splits)
# Controlled SRM grid: keep the previous no-clip baseline and test the robust z-clip candidate found in optimisation.
SRM_RIDGE_GRID = [0.0]
SRM_COVARIANCE_SHRINKAGE_GRID = [0.35, 0.40, 0.45]
SRM_Z_CLIP_GRID = [None, 2.75, 3.0, 3.25]
optimization_rows = []
print({
    "selection_method": selection_method,
    "selection_k": selection_k,
    "srm_cv_n_splits": CV_N_SPLITS,
    "srm_ridge_grid": SRM_RIDGE_GRID,
    "srm_covariance_shrinkage_grid": SRM_COVARIANCE_SHRINKAGE_GRID,
    "srm_z_clip_grid": SRM_Z_CLIP_GRID,
})


{'selection_method': 'none', 'selection_k': 8, 'srm_cv_n_splits': 5, 'srm_ridge_grid': [0.0], 'srm_covariance_shrinkage_grid': [0.35, 0.4, 0.45], 'srm_z_clip_grid': [None, 2.75, 3.0, 3.25]}


## 2. SRM Global Linear

The SRM Global Linear model learns one imaging weight vector:

```text
w = solve(cov(delta) + ridge I, mean(delta))
score = X @ w
```

Covariance shrinkage blends the empirical covariance with a simpler diagonal or identity-like estimate. This reduces sensitivity to noisy correlations when the number of imaging features is large relative to the number of participants.


In [8]:
import time

srm_trials = []
for z_clip in SRM_Z_CLIP_GRID:
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID:
        for ridge in SRM_RIDGE_GRID:
            start = time.time()
            res = srm_global_loocv(
                long_df,
                imaging_cols,
                subject_col=subject_col,
                visit_col="visit",
                selection_method=selection_method,
                k=selection_k,
                ridge=ridge,
                covariance_shrinkage=covariance_shrinkage,
                z_clip=z_clip,
                cv_n_splits=CV_N_SPLITS,
                random_seed=RANDOM_SEED,
                split_group_col=split_group_col,
                compute_ci=False,
            )
            interval_summary = adjacent_pair_interval_effect_summary(
                res["oof_df"],
                pair_col=subject_col,
                visit_col="visit",
                score_col="score",
                n_boot=N_BOOT,
                seed=RANDOM_SEED,
            )
            annual_diag = annual_tuning_diagnostics(interval_summary)
            res = {**res, **annual_diag}
            row = optimization_row(
                model="SRM Global Linear exploratory",
                params={
                    "ridge": ridge,
                    "covariance_shrinkage": covariance_shrinkage,
                    "z_clip": z_clip,
                    "selection_method": selection_method,
                    "regularization": "ridge_plus_covariance_shrinkage_plus_optional_z_clip",
                    "trial_id": len(srm_trials),
                },
                result=res,
                runtime_sec=time.time() - start,
                notes="exploratory grid tuned on mean annual d_z from V1->V2 and V2->V3; interval gap used as first tie-breaker",
            )
            srm_trials.append((res, row, interval_summary))
            optimization_rows.append(row)

srm_optimization_df = optimization_log([row for _, row, _ in srm_trials], sort_by="mean_validation_annual_dz")
print("SRM tuning candidates evaluated:", len(srm_optimization_df))

srm_review = tuning_recommendation(srm_optimization_df)
print("Numerically best SRM configuration")
display(pd.DataFrame([srm_review["raw_best"]]))
print("One-SE / near-optimal SRM candidate count:", len(srm_review["near_optimal"]))
print("Recommended SRM configuration by implemented hierarchy")
display(pd.DataFrame([srm_review["recommended"]]))
print(srm_review["summary"])
print("Human-verification summary")
display(tuning_verification_summary(srm_review))
srm_nested_candidates = [
    {
        "ridge": ridge,
        "covariance_shrinkage": covariance_shrinkage,
        "z_clip": z_clip,
        "selection_method": selection_method,
        "k": selection_k,
    }
    for z_clip in SRM_Z_CLIP_GRID
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID
    for ridge in SRM_RIDGE_GRID
]
# Use the annual-consistency hierarchy to identify the displayed candidate region.
srm_choice_table = srm_optimization_df.rename(columns={"param_k": "feature_count"}).copy()
srm_choice_table["feature_count"] = len(imaging_cols)
srm_choice_table["se_validation_dz"] = (srm_choice_table["d_ci_high"] - srm_choice_table["d_ci_low"]) / (2 * 1.96)
srm_choice_table["se_validation_dz"] = srm_choice_table["se_validation_dz"].replace([np.inf, -np.inf], np.nan).fillna(0.05)
srm_choice_table["jaccard_stability"] = 1.0
srm_choice = select_hierarchical_candidate(srm_choice_table)
print("Exploratory annual-consistency choice")
display(pd.DataFrame([srm_choice]))

start = time.time()
global_res = srm_global_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=srm_nested_candidates,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=5,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=True,
    tuning_metric="annual_mean_dz",
)
global_intervals = adjacent_pair_interval_effect_summary(
    global_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
global_annual_diag = annual_tuning_diagnostics(global_intervals)
global_res = {**global_res, **global_annual_diag}
nested_row = optimization_row(
    model="SRM Global Linear nested",
    params={"candidate_count": len(srm_nested_candidates), "inner_folds": 5, "tuning": "train-fold inner grouped CV; final reporting uses annual interval diagnostics"},
    result=global_res,
    runtime_sec=time.time() - start,
    notes="nested estimate with annual interval diagnostics: mean annual d_z plus interval gap",
)
optimization_rows.append(nested_row)
display(global_res["chosen_params_df"].head())
display(global_intervals)
pd.DataFrame([{
    "model": "SRM Global Linear nested",
    "selection_method": selection_method,
    "candidate_count": len(srm_nested_candidates),
    "dz_v1_v2": global_res["dz_v1_v2"],
    "dz_v2_v3": global_res["dz_v2_v3"],
    "mean_annual_d_z": global_res["mean_validation_annual_dz"],
    "annual_interval_gap": global_res["annual_interval_gap"],
    "p_progression": global_res["p_progression"],
    "pooled_pair_d_z_reference": global_res["d_score"],
    "ci_low": global_res["d_ci_low"],
    "ci_high": global_res["d_ci_high"],
    "n_subject_pairs": global_res["n_subjects"],
}])


SRM tuning candidates evaluated: 12
Numerically best SRM configuration


,param_ridge,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,param_trial_id,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank
0,0.0,0.4,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,4,1.12131,0.77849,0.9499,0.34282,0.819024,NaN,NaN,NaN,NaN,NaN,1.0


One-SE / near-optimal SRM candidate count: 1
Recommended SRM configuration by implemented hierarchy


,param_ridge,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,param_trial_id,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,se_validation_dz,raw_rank,directional_consistency,score_ranking_stability
0,0.0,0.4,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,4,1.12131,0.77849,0.9499,0.34282,0.819024,inf,-inf,-inf,-inf,NaN,1.0,-inf,-inf


Recommended candidate is also the raw best by mean annual validation d_z; the consistency, progression-probability, simplicity, and stability tie-breakers did not select a different row.
Human-verification summary


...

Exploratory annual-consistency choice


,model,feature_pool,objective,d_score,dz_v1_v2,dz_v2_v3,mean_validation_annual_dz,annual_interval_gap,p_progression,se_validation_dz,...,param_z_clip,param_selection_method,param_regularization,param_trial_id,feature_count,jaccard_stability,sign_stability,coefficient_sign_stability,directional_consistency,score_ranking_stability
1,SRM Global Linear exploratory,all_imaging,d_score,0.935013,1.112154,0.787096,0.949625,0.325058,0.813973,0.05,...,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,3,146,1.0,-inf,-inf,-inf,-inf


,outer_fold,inner_d_score,inner_tuning_metric,inner_dz_v1_v2,inner_dz_v2_v3,inner_annual_interval_gap,inner_p_progression,n_features,ridge,covariance_shrinkage,z_clip,selection_method,k
0,1,0.834700,annual_mean_dz,1.036249,0.633151,0.403099,0.806777,146,0.0,0.40,NaN,none,8
1,2,0.932783,annual_mean_dz,1.283904,0.581662,0.702242,0.799048,146,0.0,0.35,2.75,none,8
2,3,0.909255,annual_mean_dz,1.123496,0.695014,0.428481,0.832773,146,0.0,0.35,2.75,none,8
3,4,0.838641,annual_mean_dz,0.993804,0.683477,0.310327,0.782040,146,0.0,0.35,NaN,none,8
4,5,0.836501,annual_mean_dz,1.111690,0.561312,0.550379,0.795679,146,0.0,0.35,NaN,none,8


,interval,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V2,108,2.054886,1.819792,1.129187,0.928324,1.404750,0.888889
1,V2->V3,99,1.804783,2.314215,0.779868,0.582348,1.021959,0.757576


,model,selection_method,candidate_count,dz_v1_v2,dz_v2_v3,mean_annual_d_z,annual_interval_gap,p_progression,pooled_pair_d_z_reference,ci_low,ci_high,n_subject_pairs
0,SRM Global Linear nested,none,12,1.129187,0.779868,0.954528,0.349319,0.823232,0.935054,0.794614,1.100325,207


## 3. Clinical Benchmark Table

This final table compares model `d_z` and bootstrap confidence intervals with FARS, SARA, and the top single imaging feature. Clinical rows use only adjacent visit changes such as `FARS2-FARS1` and `FARS3-FARS2`.


In [9]:
model_rows = pd.DataFrame([
    {"feature": "SRM Global Linear", "kind": "model", "d": global_res["d_score"], "ci_low": global_res["d_ci_low"], "ci_high": global_res["d_ci_high"]},
])
ref = benchmark_table("placeholder", np.nan, np.nan, np.nan).iloc[1:]
display(pd.concat([model_rows, ref], ignore_index=True))


,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,SRM Global Linear,model,0.935054,0.794614,1.100325,NaN,NaN
1,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
2,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
3,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN


## 4. Final Framework Metrics and Clinical Validation


In [10]:
# Display interval-aware final metrics from the actual pair-table OOF model scores.
# Final evaluation: V1->V2 and V2->V3 are primary annual intervals; V1->V3 is secondary cumulative 24-month change.
subject_imaging_cols = [c for c in imaging_cols if not subject_long_df.empty and c in subject_long_df.columns]
annual_interval_summary = adjacent_pair_interval_effect_summary(
    global_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
pooled_annual_summary = pooled_adjacent_pair_effect_summary(
    global_res["oof_df"],
    pair_col=subject_col,
    visit_col="visit",
    score_col="score",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
oof_scores = global_res["oof_df"].copy()
parsed_pairs = oof_scores[subject_col].astype(str).str.extract(r"(?P<subject_id>.+)_(?P<pair_type>V1V2|V2V3)$")
oof_scores = pd.concat([oof_scores, parsed_pairs], axis=1)
v1_scores = oof_scores.loc[oof_scores["pair_type"].eq("V1V2") & oof_scores["visit"].eq(1), ["subject_id", "score"]].rename(columns={"score": "score_v1"})
v3_scores = oof_scores.loc[oof_scores["pair_type"].eq("V2V3") & oof_scores["visit"].eq(2), ["subject_id", "score"]].rename(columns={"score": "score_v3"})
v13_pairs = v1_scores.merge(v3_scores, on="subject_id", how="inner")
v13_long = pd.concat(
    [
        v13_pairs[["subject_id", "score_v1"]].rename(columns={"score_v1": "score"}).assign(visit=1, time_years=0.0),
        v13_pairs[["subject_id", "score_v3"]].rename(columns={"score_v3": "score"}).assign(visit=3, time_years=2.0),
    ],
    ignore_index=True,
)
v13_summary = interval_effect_summary(
    v13_long,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    time_col="time_years",
    intervals=[(1, 3, "V1->V3", False)],
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
framework_summary = pd.concat([annual_interval_summary, pooled_annual_summary, v13_summary], ignore_index=True, sort=False)
framework_summary = framework_summary.rename(columns={"n_pairs": "n_subject_pairs", "d_z_ci_low": "ci_low", "d_z_ci_high": "ci_high"})
framework_summary.insert(0, "model", "SRM Global Linear")
framework_summary["oof"] = True

srm_dz_summary = framework_summary[["interval", "n_subject_pairs", "d_z", "ci_low", "ci_high", "p_delta_positive"]].copy()
print("SRM OOF d_z summary")
display(srm_dz_summary)

if subject_imaging_cols:
    single_feature_baselines = single_feature_interval_baselines(
        subject_long_df,
        subject_imaging_cols,
        subject_col="subject_id",
        visit_col="visit",
        time_col="time_years",
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
else:
    single_feature_baselines = pd.DataFrame()

clinical_vars = [c for c in ["FARS", "SARA", "mfars_total", "sara_total"] if c in subject_long_df.columns]
if clinical_vars and subject_imaging_cols:
    clinical_interval_rows = []
    for clinical_col in clinical_vars:
        tmp = interval_effect_summary(
            subject_long_df.dropna(subset=[clinical_col]),
            subject_col="subject_id",
            visit_col="visit",
            score_col=clinical_col,
            time_col="time_years",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        ).assign(feature=clinical_col, kind="clinical_scale")
        clinical_interval_rows.append(tmp)
    clinical_intervals = pd.concat(clinical_interval_rows, ignore_index=True)
else:
    clinical_intervals = pd.DataFrame()

clinical_validity_table = pd.DataFrame()

if not framework_summary.empty:
    composite_intervals = framework_summary.rename(columns={"ci_low": "d_z_ci_low", "ci_high": "d_z_ci_high"})[
        ["interval", "n_subject_pairs", "d_z", "d_z_ci_low", "d_z_ci_high", "p_delta_positive"]
    ].rename(columns={"n_subject_pairs": "n_pairs"})
    final_matrix = final_model_performance_matrix(
        composite_intervals=composite_intervals,
        clinical_intervals=clinical_intervals,
        single_feature_intervals=single_feature_baselines,
        clinical_validity=clinical_validity_table,
    )
    performance_rows = assemble_performance_rows(
        "SRM Global Linear",
        composite_intervals=composite_intervals,
        clinical_intervals=clinical_intervals,
        single_feature_intervals=single_feature_baselines,
        clinical_validity=clinical_validity_table,
        cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
        source="srm_composite.ipynb",
    )
    performance_csv = save_one_performance_csv(performance_rows, REPO_ROOT / "results" / "model_performance_summary.csv")
    print("Final model performance matrix")
    display(final_matrix)
    print(f"Saved one consolidated model-performance CSV: {performance_csv}")
    display(performance_rows)


SRM OOF d_z summary


,interval,n_subject_pairs,d_z,ci_low,ci_high,p_delta_positive
0,V1->V2,108,1.129187,0.928324,1.404750,0.888889
1,V2->V3,99,0.779868,0.582348,1.021959,0.757576
2,V1->V2 + V2->V3,207,0.935054,0.784286,1.121321,0.826087
3,V1->V3,90,1.457669,1.241027,1.791011,0.955556


Final model performance matrix


,question,metric,role,value
0,12-month sensitivity V1->V2,V1->V2 paired d_z,Primary,1.129187
1,12-month sensitivity V2->V3,V2->V3 paired d_z,Primary temporal replication,0.779868
2,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,1.457669
3,Direction consistency,P(delta > 0),Secondary,0.8888888888888888; 0.7575757575757576
4,Robustness,Bootstrap CI for d_z,Primary uncertainty,"V1->V2 [0.9283238844050731, 1.4047503464333035..."
5,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,1.4576691530839114 vs 0.728521346993977
6,Better than MRI alone?,vs strongest individual MRI feature,RQ1,1.4576691530839114 vs -0.9768059805067008
7,Disease specific?,FRDA vs control change,Specificity,NaN
8,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN
9,Tracks clinical change?,Spearman delta Z vs delta FARS/SARA,Strong RQ3,NaN


Saved one consolidated model-performance CSV: /Users/robertwang/Documents/New_project/biomarkers/results/model_performance_summary.csv


,model,question,metric,role,value,n,status,evidence,cv_mode,source
0,SRM Global Linear,12-month sensitivity V1->V2,"V1->V2 paired d_z, CI, N, P(delta>0)",Primary,"1.1291871274790128 [0.9283238844050731, 1.4047...",108.0,computed,composite V1->V2 OOF annual interval,subject-level grouped 5-fold,srm_composite.ipynb
1,SRM Global Linear,12-month sensitivity V2->V3,"V2->V3 paired d_z, CI, N, P(delta>0)",Primary temporal replication,"0.7798683090559383 [0.5823482827310947, 1.0219...",99.0,computed,composite V2->V3 OOF annual interval,subject-level grouped 5-fold,srm_composite.ipynb
2,SRM Global Linear,12-month pooled annual sensitivity,"Pooled V1->V2 + V2->V3 paired d_z, CI, N, P(de...",Pooled annual diagnostic,"0.9350539070265605 [0.7842858112139988, 1.1213...",207.0,computed,pooled OOF annual pair deltas; participant-gro...,subject-level grouped 5-fold,srm_composite.ipynb
3,SRM Global Linear,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,1.457669,90.0,computed,composite V1->V3 cumulative,subject-level grouped 5-fold,srm_composite.ipynb
4,SRM Global Linear,Direction consistency,P(delta > 0),Secondary,0.8888888888888888; 0.7575757575757576,108.0,computed,annual V1->V2 and V2->V3 P(delta>0),subject-level grouped 5-fold,srm_composite.ipynb
5,SRM Global Linear,Robustness,bootstrap CI for d_z,Primary uncertainty,"V1->V2 [0.9283238844050731, 1.4047503464333035...",108.0,computed,annual interval bootstrap CI,subject-level grouped 5-fold,srm_composite.ipynb
6,SRM Global Linear,Better than clinical scale?,d_z composite vs FARS/SARA,RQ1,1.4576691530839114 vs 0.728521346993977,90.0,computed,clinical interval benchmark,subject-level grouped 5-fold,srm_composite.ipynb
7,SRM Global Linear,Better than MRI alone?,vs strongest individual MRI feature,RQ1,1.4576691530839114 vs cerebellumFS: -0.9768059...,90.0,computed,strongest single MRI feature,subject-level grouped 5-fold,srm_composite.ipynb
8,SRM Global Linear,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,missing,FRDA vs control change,subject-level grouped 5-fold,srm_composite.ipynb
9,SRM Global Linear,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN,NaN,missing,cross-sectional Spearman,subject-level grouped 5-fold,srm_composite.ipynb
